# 03 — Allocation diagnostic

Run all five diagnostic sections (combined-book exposure, correlation,
redundancy, risk contribution, benchmark comparison) on the resolved book
and export `reports/allocation_diagnostic.html`.


In [1]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.diagnostic import (
    benchmark_comparison, combined_exposure, combined_exposure_figure,
    correlation_figure, correlation_matrix, redundancy_pairs,
    render_html_report, risk_contribution,
)
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.data.providers import YahooFinanceProvider

from hailmary.allocation.returns import last_business_day_on_or_before

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/allocation_diagnostic.html')
START = date(2022, 1, 1)
END = last_business_day_on_or_before(date.today())
REDUNDANCY_THRESHOLD = 0.85

# ALIGN_WINDOW = True  (default, recommended)
#   Combined-book metrics + windowed table use the common-history window
#   (latest first-data date across all HOLDING portfolios). Apples-to-apples,
#   shorter history, static weights.
# ALIGN_WINDOW = False
#   Full available history with dynamic per-timestep weight renormalisation.
#   Longer history but early dates use only the older portfolios at boosted
#   weights, so 'early book' ≠ 'today's book'. Useful if you want pre-2024
#   context at the cost of mixing weight regimes.
ALIGN_WINDOW = True

# Annual-return target for the traffic-light styling on the benchmark table's
# 'Ann. return' column. Green ≥ target, yellow [0..target), red < 0.
# High-Sharpe-low-return rows (e.g. Simple SGD with Sharpe ~4 and ann return 1.5%)
# will be yellow under a 5% target — they're risk-adjusted-great but won't grow
# wealth at your target rate.
TARGET_ANN_RETURN = 0.05

# Date the portfolio-reconciliation section projects to. Defaults to the last
# business day. Stashaway's app values are sometimes delayed by a day —
# if the app shows 'as of 22 May' while today is 24 May, set this to
# date(2026, 5, 22) to get an exact apples-to-apples reconcile.
RECONCILE_AS_OF = END

print(f'window: {START}..{END}  |  align_window={ALIGN_WINDOW}  |  target_ann_return={TARGET_ANN_RETURN:.1%}  |  reconcile_as_of={RECONCILE_AS_OF}')

window: 2022-01-01..2026-05-22  |  align_window=True  |  target_ann_return=5.0%  |  reconcile_as_of=2026-05-22


## Parse + tag + fetch returns

In [2]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
print(f'Resolved {len(portfolios)} portfolios; fetched {returns.shape[1]} ticker series')

2026-05-24 23:29:30.210 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 49 symbols from Yahoo (2022-01-01 → 2026-05-22); inclusive


2026-05-24 23:29:33.538 | DEBUG    | hailmary.data.cache:set:45 - Cached 53806 rows key=1278c92b99d6


Resolved 15 portfolios; fetched 49 ticker series


## Fetch USDSGD (daily series for return FX adjustment)

In [3]:
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
# Yahoo's USDSGD is labelled in UK time, so its 'closing' rate lands ~7h
# before Stashaway's Singapore-EOD snapshot. For statement-date AUM we use
# the rate parsed from the PDF (attached to portfolio.metadata via from_parsed).
# Daily series is still used for compounding return adjustments — day-over-day
# moves are roughly the same despite the timezone shift.
stashaway_fx = portfolios[0].metadata.get('statement_fx_usd_sgd')
yahoo_spot = float(fx_series_usd_sgd.iloc[-1])
print('Statement-date FX (from PDF, used for AUM):  1 USD = {:.4f} SGD'.format(stashaway_fx))
print('Yahoo USDSGD spot ({}, used for daily series): 1 USD = {:.4f} SGD'.format(
    fx_series_usd_sgd.index.max().date(), yahoo_spot,
))

2026-05-24 23:29:33.559 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=d0953532618a


Statement-date FX (from PDF, used for AUM):  1 USD = 1.2732 SGD
Yahoo USDSGD spot (2026-05-22, used for daily series): 1 USD = 1.2777 SGD


## Validation — no portfolio silently dropped + window summary

In [4]:
from hailmary.allocation.diagnostic import _build_returns_panel, PortfolioDroppedError
try:
    _validation_panel = _build_returns_panel(
        holding, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd, strict=True,
    )
    print(f'All {len(holding)} HOLDING portfolios resolved cleanly '
          f'({_validation_panel.shape[1]} series, {_validation_panel.shape[0]:,} dates).')
except PortfolioDroppedError as exc:
    print('FAIL — would drop portfolios:')
    for name, reason in exc.dropped:
        print(f'  {name}: {reason}')
    raise

first_dates = (
    _validation_panel.apply(lambda c: c.dropna().index.min().date())
    .sort_values(ascending=False)
)
common_start = first_dates.iloc[0]
aligned_days = len(_validation_panel.loc[str(common_start):].dropna(how='any'))
print()
print(f'Common-history window: [{common_start}..{END}] ({aligned_days:,} aligned dates).')
print('Per-portfolio first-data dates (latest first — these are what shrink the window):')
for name, first_date in first_dates.head(8).items():
    marker = '  ← constrains common_start' if first_date == common_start else ''
    print(f'  {name:<22} {first_date}{marker}')
if len(first_dates) > 8:
    remaining = first_dates.iloc[8:]
    print(f'  ...{len(remaining)} more portfolios start between '
          f'{remaining.min()} and {remaining.max()}')
print()
print('Set ALIGN_WINDOW=True (default) → combined-book metrics use this aligned window.')
print('Set ALIGN_WINDOW=False → full per-portfolio histories with dynamic-renorm weights.')

All 15 HOLDING portfolios resolved cleanly (15 series, 1,602 dates).

Common-history window: [2024-10-30..2026-05-22] (278 aligned dates).
Per-portfolio first-data dates (latest first — these are what shrink the window):
  Nasdaq Covered Call    2024-10-30  ← constrains common_start
  Singapore Investing    2024-01-23
  BlackRock              2022-01-05
  General Investing      2022-01-05
  General SRS            2022-01-05
  Energy                 2022-01-04
  Simple USD             2022-01-04
  Utilities              2022-01-04
  ...7 more portfolios start between 2022-01-02 and 2022-01-04

Set ALIGN_WINDOW=True (default) → combined-book metrics use this aligned window.
Set ALIGN_WINDOW=False → full per-portfolio histories with dynamic-renorm weights.


## Combined-book exposure

In [5]:
exposure = combined_exposure(portfolios)
for dim, df in exposure.items():
    print(f'\n--- {dim.replace("_", " ").title()} ---')
    display(df)
combined_exposure_figure(exposure)


--- Asset Class ---


,bucket,value,weight
0,Cash,2307268.54,0.781382
1,Equity,493043.15,0.166974
2,Crypto,107445.47,0.036388
3,Commodity,26155.95,0.008858
4,Bond,18893.00,0.006398



--- Region ---


,bucket,value,weight
0,Singapore,2050013.78,0.694260
1,US,540928.57,0.183191
2,Global,216117.91,0.073191
3,Developed ex-US,68464.96,0.023186
4,Japan,31107.54,0.010535
5,Emerging Markets,20305.17,0.006877
6,India,11899.27,0.004030
7,Asia ex-Japan,5590.93,0.001893
8,Emerging Markets ex-China,3256.74,0.001103
9,Eurozone,2075.77,0.000703



--- Sector ---


,bucket,value,weight
0,Money Market,2025167.63,0.685845
1,Broad Market,306149.57,0.103681
2,Treasury 0-3M (proxy: BIL),274197.24,0.092860
3,Bitcoin (proxy: BTC-USD spot),54778.04,0.018551
4,Ethereum (proxy: ETH-USD spot),52667.43,0.017836
5,Technology,31193.15,0.010564
6,Gold,26155.95,0.008858
7,Dividend,23858.50,0.008080
8,Hedged,21353.80,0.007232
9,Consumer Staples,19655.78,0.006657


## Correlation matrix

In [6]:
corr = correlation_matrix(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
display(corr.round(3))
correlation_figure(corr)

,BlackRock,Energy,General Investing,Simple USD,Singapore Investing,Utilities,Income Investing,High Dividend Yield,Ex-US Large-cap,SG ETF,Nasdaq Covered Call,Guitsa,Simple SGD,General SRS,Crypto
BlackRock,1.000,0.261,0.736,0.325,0.274,0.316,0.332,0.570,0.717,0.233,0.863,-0.001,-0.001,0.746,0.326
Energy,0.261,1.000,0.441,0.245,-0.169,0.327,0.063,0.624,0.414,-0.053,-0.033,-0.060,-0.060,0.443,0.210
General Investing,0.736,0.441,1.000,0.330,-0.072,0.472,0.412,0.831,0.898,0.036,0.415,0.030,0.030,0.999,0.730
Simple USD,0.325,0.245,0.330,1.000,-0.194,0.307,-0.016,0.363,0.321,-0.217,0.229,0.013,0.013,0.334,0.122
Singapore Investing,0.274,-0.169,-0.072,-0.194,1.000,-0.064,-0.061,-0.139,-0.046,0.800,0.361,0.024,0.024,-0.068,-0.106
Utilities,0.316,0.327,0.472,0.307,-0.064,1.000,0.396,0.662,0.470,-0.053,0.106,-0.034,-0.034,0.479,0.195
Income Investing,0.332,0.063,0.412,-0.016,-0.061,0.396,1.000,0.371,0.469,0.013,0.003,-0.044,-0.044,0.418,0.208
High Dividend Yield,0.570,0.624,0.831,0.363,-0.139,0.662,0.371,1.000,0.790,-0.028,0.209,-0.031,-0.031,0.839,0.416
Ex-US Large-cap,0.717,0.414,0.898,0.321,-0.046,0.470,0.469,0.790,1.000,0.082,0.280,-0.002,-0.002,0.908,0.467
SG ETF,0.233,-0.053,0.036,-0.217,0.800,-0.053,0.013,-0.028,0.082,1.000,0.321,0.019,0.019,0.039,-0.038


## Redundancy (threshold default 0.85)

In [7]:
pairs = redundancy_pairs(corr, threshold=REDUNDANCY_THRESHOLD, portfolios=portfolios)
if pairs:
    pd.DataFrame(pairs, columns=['a', 'b', 'rho', 'candidate'])
else:
    print(f'No portfolio pairs above ρ = {REDUNDANCY_THRESHOLD}.')
    print('If your customs are uncorrelated by design, this is expected — drop the threshold to 0.7 to surface near-redundancy.')

## Risk contribution

In [8]:
risk = risk_contribution(portfolios, returns=returns)
print('--- By portfolio ---')
display(risk['by_portfolio'].round(4))
print('--- By holding (top 15) ---')
display(risk['by_holding'].head(15).round(4))

--- By portfolio ---


,name,weight,value,contribution,pct_total,delta_vol
0,General Investing,0.1258,371548.08,0.0195,0.5555,0.0168
1,Crypto,0.0186,54802.94,0.0075,0.2151,0.0064
2,General SRS,0.0343,101299.95,0.0052,0.1476,0.0041
3,High Dividend Yield,0.0081,23860.69,0.0009,0.0257,0.0006
4,BlackRock,0.0133,39180.06,0.0008,0.0240,0.0004
5,Ex-US Large-cap,0.0027,7864.31,0.0004,0.0108,0.0003
6,Simple SGD,0.6612,1952415.36,0.0003,0.0074,-0.0679
7,Energy,0.0028,8220.98,0.0002,0.0065,0.0001
8,Nasdaq Covered Call,0.0027,7890.77,0.0001,0.0042,0.0001
9,Utilities,0.0013,3913.16,0.0001,0.0025,0.0000


--- By holding (top 15) ---


,holding,weight,value,contribution,pct_total,delta_vol
0,Crypto · ETH-USD,0.0090,26531.45,0.0046,0.1310,0.0041
1,General Investing · ETH-USD,0.0072,21233.09,0.0037,0.1048,0.0033
2,Crypto · BTC-USD,0.0094,27723.36,0.0029,0.0842,0.0025
3,General Investing · BTC-USD,0.0073,21660.01,0.0023,0.0658,0.0020
4,General Investing · VEU,0.0160,47157.40,0.0023,0.0648,0.0017
5,General Investing · IVV,0.0149,43933.12,0.0022,0.0629,0.0017
6,General Investing · XLK,0.0083,24451.16,0.0018,0.0502,0.0015
7,General Investing · ISAC.L,0.0204,60101.90,0.0015,0.0437,0.0007
8,High Dividend Yield · VYM,0.0081,23858.50,0.0009,0.0257,0.0006
9,General Investing · RSP,0.0068,20045.53,0.0009,0.0250,0.0006


## Benchmark comparison

In [9]:
bench = benchmark_comparison(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
bench.round(3)

,value,total_return,ann_return,ytd_return,sharpe,max_dd,annualised_vol,n_days,sharpe_delta_vs_BlackRock,max_dd_delta_vs_BlackRock,vol_delta_vs_BlackRock,sharpe_delta_vs_General Investing,max_dd_delta_vs_General Investing,vol_delta_vs_General Investing,sharpe_delta_vs_Singapore Investing,max_dd_delta_vs_Singapore Investing,vol_delta_vs_Singapore Investing,sharpe_delta_vs_Income Investing,max_dd_delta_vs_Income Investing,vol_delta_vs_Income Investing
Combined book,3200757.553,0.021,0.019,0.006,0.427,-0.040,0.046,278.0,-0.070,0.079,-0.080,0.224,0.104,-0.122,-2.620,-0.015,0.008,-0.527,-0.006,0.001
BlackRock,50061.538,0.062,0.056,0.054,0.497,-0.118,0.125,278.0,0.000,0.000,0.000,0.294,0.025,-0.042,-2.549,-0.093,0.088,-0.457,-0.084,0.080
Energy,10504.193,0.117,0.105,0.164,0.510,-0.225,0.266,278.0,0.013,-0.106,0.141,0.307,-0.081,0.098,-2.537,-0.199,0.228,-0.444,-0.191,0.221
General Investing,474738.124,0.022,0.020,0.018,0.203,-0.143,0.168,278.0,-0.294,-0.025,0.042,0.000,0.000,0.000,-2.844,-0.118,0.130,-0.751,-0.109,0.122
Simple USD,350350.816,0.018,0.016,-0.002,0.315,-0.050,0.055,278.0,-0.183,0.068,-0.070,0.111,0.093,-0.112,-2.732,-0.025,0.018,-0.639,-0.016,0.010
Singapore Investing,19979.290,0.134,0.121,0.043,3.047,-0.025,0.038,278.0,2.549,0.093,-0.088,2.844,0.118,-0.130,0.000,0.000,0.000,2.093,0.009,-0.008
Utilities,4999.962,0.125,0.113,0.079,0.688,-0.128,0.179,278.0,0.190,-0.009,0.054,0.485,0.016,0.011,-2.359,-0.102,0.141,-0.266,-0.094,0.134
Income Investing,5015.850,0.048,0.043,-0.008,0.954,-0.034,0.045,278.0,0.457,0.084,-0.080,0.751,0.109,-0.122,-2.093,-0.009,0.008,0.000,0.000,0.000
High Dividend Yield,30487.519,0.074,0.067,0.066,0.480,-0.151,0.162,278.0,-0.017,-0.032,0.036,0.277,-0.008,-0.006,-2.567,-0.126,0.124,-0.474,-0.117,0.116
Ex-US Large-cap,10048.465,0.145,0.131,0.023,0.764,-0.130,0.183,278.0,0.267,-0.011,0.057,0.561,0.013,0.015,-2.283,-0.105,0.145,-0.190,-0.096,0.138


## Export HTML report (strict — fails if any portfolio would drop)

In [10]:
out = render_html_report(
    portfolios,
    REPORT_PATH,
    returns=returns,
    redundancy_threshold=REDUNDANCY_THRESHOLD,
    fx_series_usd_sgd=fx_series_usd_sgd,
    align_window=ALIGN_WINDOW,
    target_ann_return=TARGET_ANN_RETURN,
    reconciliation_as_of=RECONCILE_AS_OF,
    title='Stashaway book — allocation diagnostic',
)  # fx_rate_usd_sgd left default → picks up Stashaway PDF rate from portfolio.metadata
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\allocation_diagnostic.html
